# LoRA Fine-tuning of GPT-2 M on DART

Reproduction of LoRA (Hu et al., 2021) on **DART: Open-Domain Structured Data Record to Text Generation** (Nan et al., 2021).

- **Backbone:** GPT-2 Medium (frozen)
- **Adapters:** LoRA on attention projections (default targets: `q`, `v`)
- **Task:** DART v1.1.1 — the version the LoRA paper used
- **Separator:** `<|SEP|>` added as a single special token
- **Metrics:** BLEU, NIST, METEOR, ROUGE-L, CIDEr

## Hyperparameters — the paper's DART LoRA recipe

Sourced from the paper's official training command in `microsoft/LoRA/examples/NLG/src/gpt2_ft.py` for DART:

```bash
python src/gpt2_ft.py \
    --train_batch_size 8 --grad_acc 1 --seq_len 512 \
    --lr 0.0002 --weight_decay 0.00 --adam_beta2 0.999 \
    --clip 0.0 --scheduler linear --warmup_step 500 \
    --max_epoch 5 --label_smooth 0.0 --random_seed 110 \
    --lora_dim 4 --lora_alpha 32 --lora_dropout 0.1
```

Loss is over the full sequence (paper does no `-100` masking on the prompt). Training in fp16. Inference: beam=10, length_penalty=0.8, no_repeat_ngram_size=4.

Switch ablations by editing `CFG` (`lora_rank`, `lora_targets`). Outputs go to `results/lora_dart_v1.1.1/`.

## 1. Setup

In [ ]:
# One-time installs (uncomment on first run)
# %pip install torch transformers accelerate sacrebleu nltk rouge-score pycocoevalcap tqdm
# import nltk; nltk.download('wordnet'); nltk.download('omw-1.4'); nltk.download('punkt'); nltk.download('punkt_tab')

In [ ]:
import os, json, math, random, time, urllib.request, sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup, AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

from dart_loader import load_dart

SEED = 110
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
CFG = {
    'model_name': 'gpt2-medium',
    'data_dir': '../data/raw/dart',
    'data_version': 'v1.1.1',        # DART version used by the LoRA paper
    # --- LoRA (paper) ---
    'lora_rank': 4,                  # --lora_dim 4
    'lora_alpha': 32,                # --lora_alpha 32
    'lora_dropout': 0.0,             # DART differs from other datsets
    'lora_targets': ['q', 'v'],      # subset of {'q','k','v','o'}
    # --- Optimization (paper: src/gpt2_ft.py for DART LoRA) ---
    'max_length': 512,               # --seq_len 512
    'batch_size': 8,                 # --train_batch_size 8
    'grad_accum': 1,                 # --grad_acc 1
    'epochs': 5,                     # --max_epoch 5
    'lr': 2e-4,                      # --lr 0.0002
    'weight_decay': 0.0,             # DART differs from other datsets
    'adam_beta1': 0.9,               # AdamW default
    'adam_beta2': 0.999,             # --adam_beta2 0.999
    'adam_eps': 1e-8,                # AdamW default
    'warmup_steps': 500,             # --warmup_step 500
    'label_smoothing': 0.0,          # DART differs from other datsets
    'grad_clip_norm': None,          # --clip 0.0 (no clipping)
    'use_fp16': True,                # paper trains in fp16
    'mask_prompt': True,             # mask up to & including BOS so loss is target-only (paper: LineByLineTriplesTextDataset)
    # --- Inference (paper Table; DART row) ---
    'beam_size': 10,
    'length_penalty': 0.8,
    'no_repeat_ngram_size': 4,
    'max_new_tokens': 100,
    'out_dir': Path('../results/lora_dart_v1.1.1'),
}
CFG['out_dir'].mkdir(parents=True, exist_ok=True)
Path(CFG['data_dir']).mkdir(parents=True, exist_ok=True)
print('Run dir:', CFG['out_dir'].resolve())

## 2. Load DART (v1.1.1)

Downloads the official JSON splits from the Yale-LILY DART GitHub repo on first call (cached afterward). We avoid Hugging Face `datasets` so this works on `datasets>=4.0` and on Windows installs without long-path support.

Each DART entry has a `tripleset` (list of `[subj, pred, obj]`) and `annotations` (list of `{source, text}` references). We serialize the tripleset as `subj : pred : obj | ...` to match the paper's `format_converting_dart.py` preprocessing.

Expected sizes for v1.1.1: train=62,659, dev=2,768, test=5,097.

In [ ]:
train_raw, dev_raw, test_raw = load_dart(CFG['data_dir'], version=CFG['data_version'])
print(f"version={CFG['data_version']}  train={len(train_raw)}  dev={len(dev_raw)}  test={len(test_raw)}")
print('Sample:', train_raw[0])

In [ ]:
# Training expands one (src, ref) pair per annotation; eval keeps all references per src.
train_rows = [{'src': r['src'], 'tgt': t} for r in train_raw for t in r['refs']]
dev_rows   = dev_raw
test_rows  = test_raw
print(f'train pairs={len(train_rows)}  dev examples={len(dev_rows)}  test examples={len(test_rows)}')
print('Train sample:', train_rows[0])
print('Test  sample:', test_rows[0])

## 3. Model + LoRA Adapters

GPT-2 attention uses `Conv1D` (HF's transposed-linear). The c_attn layer outputs a packed `[Q | K | V]` of shape `(..., 3*hidden)`. We wrap it and add LoRA only to the requested slices. The output projection `c_proj` is wrapped separately when `'o'` is in targets.

In [ ]:
DELIMITER = '<|SEP|>'

tokenizer = GPT2Tokenizer.from_pretrained(CFG['model_name'])
print('Tokenizer len before:', len(tokenizer))
n_old = len(tokenizer)
tokenizer.add_special_tokens({'additional_special_tokens': [DELIMITER]})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
print('Tokenizer len after :', len(tokenizer))
DELIMITER_ID = tokenizer.convert_tokens_to_ids(DELIMITER)
assert DELIMITER_ID != tokenizer.unk_token_id, 'delimiter not registered'

model = GPT2LMHeadModel.from_pretrained(CFG['model_name'])
with torch.no_grad():
    model.resize_token_embeddings(len(tokenizer))
    # LoRA freezes embeddings, so init the new row from the mean of existing rows
    # instead of leaving it random.
    emb = model.get_input_embeddings().weight
    emb[n_old:] = emb[:n_old].mean(dim=0, keepdim=True)
model.config.pad_token_id = tokenizer.pad_token_id
HIDDEN = model.config.hidden_size
print('Hidden size:', HIDDEN, ' Layers:', model.config.n_layer)

In [ ]:
sys.path.append(os.path.abspath('..'))
from lora_module import inject_lora

inject_lora(model, CFG)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)')
model.to(DEVICE);

## 4. Tokenization & DataLoader

Format matches the paper's `LineByLineTriplesTextDataset` preprocessing:

```
 <linearized triples> <|SEP|> <target sentence> <|endoftext|>
```

i.e. `' {src} {bos_tok} {tgt} {eos_tok}'` — single spaces between every part, leading space included. GPT-2's `bos_token` and `eos_token` are both `<|endoftext|>`.


Loss masking (`CFG['mask_prompt']`):
- `True` (paper recipe): the BOS position is located in the tokenized sequence and labels up to and including the BOS are set to `-100`, so loss only counts target tokens
- `False`: full-sequence loss

In [ ]:
class DARTDataset(Dataset):
    """Format: ' {src} {DELIMITER} {tgt} {eos_tok}' (single spaces, leading space included).
    DELIMITER ('<|SEP|>') is a learned special token added to the tokenizer.
    When mask_prompt is True, labels up to and including the DELIMITER index are set to -100
    so loss is computed only on target tokens.
    """
    def __init__(self, rows, tok, max_len, mask_prompt):
        self.rows, self.tok, self.max_len, self.mask_prompt = rows, tok, max_len, mask_prompt
        self.sep_id = tok.convert_tokens_to_ids(DELIMITER)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        eos = self.tok.eos_token
        full = ' {} {} {} {}'.format(r['src'], DELIMITER, r['tgt'], eos)
        full_ids = self.tok(full, add_special_tokens=False, truncation=True, max_length=self.max_len)['input_ids']
        labels = list(full_ids)
        if self.mask_prompt:
            try:
                sep_pos = full_ids.index(self.sep_id)
                for j in range(sep_pos + 1):
                    labels[j] = -100
            except ValueError:
                pass
        return {'input_ids': full_ids, 'labels': labels}

def collate(batch, pad_id):
    L = max(len(b['input_ids']) for b in batch)
    ids, attn, lbl = [], [], []
    for b in batch:
        pad = L - len(b['input_ids'])
        ids.append(b['input_ids'] + [pad_id] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
        lbl.append(b['labels'] + [-100] * pad)
    return {
        'input_ids':      torch.tensor(ids),
        'attention_mask': torch.tensor(attn),
        'labels':         torch.tensor(lbl),
    }

train_ds = DARTDataset(train_rows, tokenizer, CFG['max_length'], CFG['mask_prompt'])
train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0,
    collate_fn=lambda b: collate(b, tokenizer.pad_token_id),
)
print(f'Train batches: {len(train_loader)}  (mask_prompt={CFG["mask_prompt"]})')

In [ ]:
print('Sample mask:')
print(train_loader.dataset[0])


## 5. Training

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(
    trainable_params,
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
    betas=(CFG['adam_beta1'], CFG['adam_beta2']),
    eps=CFG['adam_eps'],
)
total_steps = max(1, (len(train_loader) // CFG['grad_accum']) * CFG['epochs'])
scheduler   = get_linear_schedule_with_warmup(optimizer, CFG['warmup_steps'], total_steps)
loss_fn     = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'], ignore_index=-100)

use_amp = bool(CFG['use_fp16']) and DEVICE == 'cuda'
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)

log = {'loss': [], 'epoch_loss': []}
torch.cuda.reset_peak_memory_stats() if DEVICE == 'cuda' else None
t0 = time.time()
model.train()

for epoch in range(CFG['epochs']):
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{CFG['epochs']}")
    optimizer.zero_grad()
    running = []
    for i, batch in enumerate(pbar):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            shift_logits = out.logits[:, :-1, :].contiguous()
            shift_labels = batch['labels'][:, 1:].contiguous()
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        scaler.scale(loss / CFG['grad_accum']).backward()
        if (i + 1) % CFG['grad_accum'] == 0:
            if CFG['grad_clip_norm'] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, CFG['grad_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        running.append(loss.item())
        log['loss'].append(loss.item())
        pbar.set_postfix(loss=sum(running[-50:]) / min(50, len(running)))
    log['epoch_loss'].append(sum(running) / len(running))
    print(f"epoch {epoch+1} mean loss: {log['epoch_loss'][-1]:.4f}")

train_time = time.time() - t0
peak_mem = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE == 'cuda' else 0.0
samples_per_sec = (len(train_ds) * CFG['epochs']) / train_time
print(f'Train time: {train_time:.1f}s  peak mem: {peak_mem:.2f} GB  throughput: {samples_per_sec:.2f} samples/s')

In [ ]:
# Save adapter weights only (small file)
adapter_state = {n: p.detach().cpu() for n, p in model.named_parameters() if 'lora_' in n}
torch.save(adapter_state, CFG['out_dir'] / 'lora_adapter.pt')
with open(CFG['out_dir'] / 'train_log.json', 'w') as f:
    json.dump(log, f)
print('Saved adapter ->', CFG['out_dir'] / 'lora_adapter.pt')

In [ ]:
# Load from saved adapter (for inference or resuming training)
adapter_state = torch.load(CFG['out_dir'] / 'lora_adapter.pt', map_location=DEVICE)
for n, p in model.named_parameters():
    if n in adapter_state:
        p.data.copy_(adapter_state[n].to(p.device))

## 6. Generation on Test Set (beam search)

In [ ]:
model.eval()
predictions, references, sources = [], [], []
with torch.no_grad():
    for r in tqdm(test_rows, desc='generate'):
        prompt = ' {} {}'.format(r['src'], DELIMITER)
        ids = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(DEVICE)
        out = model.generate(
            **ids,
            max_new_tokens=CFG['max_new_tokens'],
            num_beams=CFG['beam_size'],
            do_sample=False,
            no_repeat_ngram_size=CFG['no_repeat_ngram_size'],
            length_penalty=CFG['length_penalty'],
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        gen = tokenizer.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        gen = gen.split('\n')[0].strip()
        predictions.append(gen)
        references.append(r['refs'])
        sources.append(r['src'])

with open(CFG['out_dir'] / 'predictions.jsonl', 'w', encoding='utf-8') as f:
    for s, p, refs in zip(sources, predictions, references):
        f.write(json.dumps({'src': s, 'pred': p, 'refs': refs}) + '\n')
print('Wrote', len(predictions), 'predictions ->', CFG['out_dir'] / 'predictions.jsonl')

In [ ]:
# Load predictions for evaluation (if not already in memory)
predictions, references = [], []
with open('../results/lora_dart_v1.1.1/predictions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        predictions.append(item['pred'])
        references.append(item['refs'])

## 7. Evaluation: BLEU, NIST, METEOR, ROUGE-L, CIDEr

In [ ]:
# !pip install evaluate
# !pip install rouge_score
# !pip install pycocoevalcap
# !pip install sacrebleu
from nltk.translate.nist_score import sentence_nist, corpus_nist
import sacrebleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

# Unified E2E metric evaluation (LoRA-style reporting)
# Expects:
# - unique_preds: List[str] generated from unique MRs
# - multi_refs: List[List[str]] references for each unique MR

# Lowercase once, up front
predictions_lc = [p.lower() if p else '' for p in predictions]
references_lc = [[t.lower() for t in refs] for refs in references]
unique_preds = predictions_lc
multi_refs = references_lc

def _safe_tokenize_text(s: str):
    return s.strip().split()

def compute_e2e_metrics(predictions, multi_references):
    """
    Compute BLEU, NIST, METEOR, ROUGE-L, and CIDEr for E2E NLG.
    Args:
        predictions: List[str], one prediction per unique MR.
        multi_references: List[List[str]], list of refs for each MR.
    Returns:
        Dict[str, float] with keys: BLEU, NIST, METEOR, ROUGE_L, CIDEr.
    """
    assert len(predictions) == len(multi_references), (
        f"Mismatched lengths: {len(predictions)} predictions vs "
        f"{len(multi_references)} reference groups"
    )
    # sacrebleu expects refs grouped by reference index: List[List[str]]
    max_refs = max(len(refs) for refs in multi_references)
    sacre_refs = []
    for ref_idx in range(max_refs):
        ref_stream = []
        for refs in multi_references:
            if ref_idx < len(refs):
                ref_stream.append(refs[ref_idx])
            else:
                ref_stream.append("")
        sacre_refs.append(ref_stream)
    bleu_score = sacrebleu.corpus_bleu(predictions, sacre_refs).score / 100.0
    nist_hyps = [_safe_tokenize_text(p) for p in predictions]
    nist_refs = [[_safe_tokenize_text(r) for r in refs] for refs in multi_references]
    nist_score = corpus_nist(nist_refs, nist_hyps)
    # Build COCO-style dicts once: id -> [captions].
    cider_hyps = {}
    for i, pred in enumerate(predictions):
        cider_hyps[i] = [pred]
    cider_refs = {}
    for i, refs in enumerate(multi_references):
        cider_refs[i] = refs
    # METEOR with multi-reference input (COCO-style scorer).
    meteor_scorer = Meteor()
    meteor_score, _ = meteor_scorer.compute_score(cider_refs, cider_hyps)
    # ROUGE-L with multi-reference input (COCO-style scorer).
    rouge_scorer = Rouge()
    rouge_l, _ = rouge_scorer.compute_score(cider_refs, cider_hyps)
    # CIDEr uses COCO-style dicts: id -> [captions].
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(cider_refs, cider_hyps)
    return {
        "BLEU": float(bleu_score),
        "NIST": float(nist_score),
        "METEOR": float(meteor_score),
        "ROUGE_L": float(rouge_l),
        "CIDEr": float(cider_score),
    }

# unique_raw_mrs, multi_refs = data_retrieval(testwref_df, unique_mr=True)
# unique_raw_mrs_dataset = TestDataset(unique_raw_mrs)
# unique_preds = batched_generate(model, tokenizer, unique_raw_mrs_dataset)
results = compute_e2e_metrics(unique_preds, multi_refs)
BLEU, NIST, METEOR, ROUGE_L, CIDEr = (
    results['BLEU'],
    results['NIST'],
    results['METEOR'],
    results['ROUGE_L'],
    results['CIDEr'],
)
for metric_name, metric_value in results.items():
    print(f"{metric_name}: {metric_value:.6f}")

In [ ]:
max_refs = max(len(r) for r in references)
refs_rect = [[r[i] if i < len(r) else r[0] for r in references] for i in range(max_refs)]

ter_obj = sacrebleu.corpus_ter(predictions, refs_rect)
TER = float(ter_obj.score)
print(f'TER     : {TER:.4f}')

In [ ]:
metrics = {
    'mode':    'lora',
    'dataset': 'dart_v1.1.1',
    'BLEU':    BLEU,
    'NIST':    NIST,
    'METEOR':  METEOR,
    'ROUGE-L': ROUGE_L,
    'CIDEr':   CIDEr,
    'efficiency': {
        'trainable_params': trainable,
        'total_params':     total,
        'trainable_pct':    100 * trainable / total,
        'train_time_sec':   train_time,
        'peak_gpu_gb':      peak_mem,
        'samples_per_sec':  samples_per_sec,
    },
    'config': {k: (str(v) if isinstance(v, Path) else v) for k, v in CFG.items()},
}
with open(CFG['out_dir'] / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))